# 01 – Exploratory Data Analysis
PII Detection Project — Group 15


In [12]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import ast
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load raw datasets

In [13]:
kaggle = pd.read_csv('../pii_dataset.csv/pii_dataset.csv', dtype=str)
ai4p   = pd.read_csv('../PII43k.csv', dtype=str, on_bad_lines='skip', engine='python')
print(f'Kaggle rows:     {len(kaggle):,}')
print(f'ai4privacy rows: {len(ai4p):,}')

Kaggle rows:     4,434
ai4privacy rows: 42,759


## 1.1 Dataset schema and sample rows

A quick look at the raw columns and a few representative rows from each dataset.

In [14]:
print("=== Kaggle PII Data Detection ===")
print(f"Shape: {kaggle.shape[0]:,} rows × {kaggle.shape[1]} columns")
print(f"Columns: {list(kaggle.columns)}\n")
kaggle.head(3)

=== Kaggle PII Data Detection ===
Shape: 4,434 rows × 16 columns
Columns: ['document', 'text', 'tokens', 'trailing_whitespace', 'labels', 'prompt', 'prompt_id', 'name', 'email', 'phone', 'job', 'address', 'username', 'url', 'hobby', 'len']



,document,text,tokens,trailing_whitespace,labels,prompt,prompt_id,name,email,phone,job,address,username,url,hobby,len
0,1073d46f-2241-459b-ab01-851be8d26436,"My name is Aaliyah Popova, and I am a jeweler ...","['My', 'name', 'is', 'Aaliyah', 'Popova,', 'an...","[True, True, True, True, True, True, True, Tru...","['O', 'O', 'O', 'B-NAME_STUDENT', 'I-NAME_STUD...",\n Aaliyah Popova is a jeweler with 13 year...,1,Aaliyah Popova,aaliyah.popova4783@aol.edu,(95) 94215-7906,jeweler,97 Lincoln Street,NaN,NaN,Podcasting,363
1,5ec717a9-17ee-48cd-9d76-30ae256c9354,"My name is Konstantin Becker, and I'm a develo...","['My', 'name', 'is', 'Konstantin', 'Becker,', ...","[True, True, True, True, True, True, True, Tru...","['O', 'O', 'O', 'B-NAME_STUDENT', 'I-NAME_STUD...",\n Konstantin Becker is a developer with 2 ...,1,Konstantin Becker,konstantin.becker@gmail.com,0475 4429797,developer,826 Webster Street,NaN,NaN,Quilting,255
2,353da41e-7799-4071-ab20-d959b362612e,"As Mieko Mitsubishi, an account manager at a p...","['As', 'Mieko', 'Mitsubishi,', 'an', 'account'...","[True, True, True, True, True, True, True, Tru...","['O', 'B-NAME_STUDENT', 'I-NAME_STUDENT', 'O',...",\n Mieko Mitsubishi is a account manager. W...,3,Mieko Mitsubishi,mieko_mitsubishi@msn.org,+27 61 222 4762,account manager,1309 Southwest 71st Terrace,NaN,NaN,Metal detecting,259


In [15]:
print("=== ai4privacy PII43k ===")
print(f"Shape: {ai4p.shape[0]:,} rows × {ai4p.shape[1]} columns")
print(f"Columns: {list(ai4p.columns)}\n")
ai4p.head(3)

=== ai4privacy PII43k ===
Shape: 42,759 rows × 4 columns
Columns: ['Template', 'Filled Template', 'Tokenised Filled Template', 'Tokens']



,Template,Filled Template,Tokenised Filled Template,Tokens
0,"In our video conference, discuss the role of e...","In our video conference, discuss the role of e...","['in', 'our', 'video', 'conference', ',', 'dis...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ..."
1,Could you draft a letter for [NAME_1] to send ...,"Could you draft a letter for Dietrich, Schulis...","['could', 'you', 'draft', 'a', 'letter', 'for'...","['O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NA..."
2,Discuss the options for [FULLNAME_1] who wants...,Discuss the options for Jeffery Pfeffer who wa...,"['discuss', 'the', 'options', 'for', 'jeff', '...","['O', 'O', 'O', 'O', 'B-FULLNAME', 'I-FULLNAME..."


### Reconstructed text with PII highlighted

Below we reconstruct the original text from tokens and **bold** every PII span so the data is visible at a glance.

In [16]:
# Color legend
legend_html = '<h4>PII Entity Color Legend</h4><p>'
for entity, col in [('PERSON','#e74c3c'), ('EMAIL','#3498db'), ('PHONE','#2ecc71'),
                     ('ADDRESS','#e67e22'), ('URL','#9b59b6'), ('ID','#f1c40f'), ('USERNAME','#1abc9c')]:
    legend_html += (f'<span style="background:{col};color:white;padding:2px 8px;'
                    f'border-radius:3px;margin-right:8px;font-size:12px">{entity}</span>')
legend_html += '</p>'
display(HTML(legend_html))

NameError: name 'HTML' is not defined

In [ ]:
from IPython.display import display, HTML

def highlight_pii(tokens_str, labels_str, max_tokens=120):
    """Reconstruct text with PII spans wrapped in colored HTML tags."""
    tokens = ast.literal_eval(tokens_str)[:max_tokens]
    labels = ast.literal_eval(labels_str)[:max_tokens]
    colors = {
        'PERSON': '#e74c3c', 'NAME_STUDENT': '#e74c3c',
        'EMAIL': '#3498db',
        'PHONE': '#2ecc71', 'PHONE_NUM': '#2ecc71',
        'ADDRESS': '#e67e22', 'STREET_ADDRESS': '#e67e22',
        'URL': '#9b59b6', 'URL_PERSONAL': '#9b59b6',
        'ID': '#f1c40f', 'ID_NUM': '#f1c40f',
        'USERNAME': '#1abc9c',
    }
    parts = []
    for tok, lbl in zip(tokens, labels):
        if lbl != 'O':
            entity = lbl.split('-', 1)[1]
            col = colors.get(entity, '#95a5a6')
            parts.append(f'<b style="background:{col};color:white;padding:1px 4px;border-radius:3px"'
                         f' title="{entity}">{tok}</b>')
        else:
            parts.append(tok)
    text = ' '.join(parts)
    if len(ast.literal_eval(tokens_str)) > max_tokens:
        text += ' …'
    return text

# Show 3 Kaggle samples
html = '<h4>Kaggle PII Data Detection — sample records</h4>'
for i, row in kaggle.sample(3, random_state=42).iterrows():
    html += f'<p style="font-size:13px;line-height:1.7"><b>Record {i}:</b> '
    html += highlight_pii(row['tokens'], row['labels']) + '</p>'

# Show 3 ai4privacy samples
html += '<h4>ai4privacy PII43k — sample records</h4>'
for i, row in ai4p.sample(3, random_state=42).iterrows():
    html += f'<p style="font-size:13px;line-height:1.7"><b>Record {i}:</b> '
    html += highlight_pii(row['Tokenised Filled Template'], row['Tokens']) + '</p>'

display(HTML(html))

## 2. Document length distributions

In [ ]:
kaggle['n_tokens'] = kaggle['tokens'].apply(lambda x: len(ast.literal_eval(x)))
ai4p['n_tokens']   = ai4p['Tokenised Filled Template'].apply(lambda x: len(ast.literal_eval(x)))

fig, axes = plt.subplots(1, 2)
axes[0].hist(kaggle['n_tokens'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Kaggle — tokens per document')
axes[0].set_xlabel('Tokens'); axes[0].set_ylabel('Count')

axes[1].hist(ai4p['n_tokens'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('ai4privacy — tokens per document')
axes[1].set_xlabel('Tokens')

plt.tight_layout()
plt.savefig('../outputs/results/eda_token_lengths.png', dpi=150)
plt.show()
print(kaggle['n_tokens'].describe().apply(lambda x: f'{x:.1f}').rename('Kaggle')
print("\n")
print(ai4p['n_tokens'].describe().apply(lambda x: f'{x:.1f}').rename('ai4privacy'))

## 3. Entity type distributions (Kaggle raw labels)

In [ ]:
def count_entities(df, label_col):
    cnt = Counter()
    for row in df[label_col]:
        for lbl in ast.literal_eval(row):
            if lbl.startswith('B-'):
                cnt[lbl[2:]] += 1
    return cnt

k_cnt = count_entities(kaggle, 'labels')
a_cnt = count_entities(ai4p, 'Tokens')

fig, axes = plt.subplots(1, 2)
for ax, cnt, title, color in zip(axes,
    [k_cnt, a_cnt],
    ['Kaggle entity counts', 'ai4privacy entity counts (top 15)'],
    ['steelblue', 'coral']):
    top = cnt.most_common(15)
    labels, values = zip(*top) if top else ([], [])
    ax.barh(labels[::-1], values[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Entity occurrences')

plt.tight_layout()
plt.savefig('../outputs/results/eda_entity_counts.png', dpi=150)
plt.show()

## 4. PII density — fraction of tokens that are PII

In [ ]:
def pii_density(df, label_col):
    densities = []
    for row in df[label_col]:
        lbls = ast.literal_eval(row)
        n_pii = sum(1 for l in lbls if l != 'O')
        densities.append(n_pii / len(lbls) if lbls else 0)
    return densities

k_dens = pii_density(kaggle, 'labels')
a_dens = pii_density(ai4p, 'Tokens')

fig, ax = plt.subplots()
ax.hist(k_dens, bins=40, alpha=0.6, label='Kaggle', color='steelblue')
ax.hist(a_dens, bins=40, alpha=0.6, label='ai4privacy', color='coral')
ax.set_xlabel('Fraction of tokens that are PII')
ax.set_ylabel('Documents')
ax.set_title('PII density per document')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/results/eda_pii_density.png', dpi=150)
plt.show()
print(f'Kaggle mean density:     {sum(k_dens)/len(k_dens)*100:.2f}%')
print(f'ai4privacy mean density: {sum(a_dens)/len(a_dens)*100:.2f}%')

## 5. Sample records

In [ ]:
for i, row in kaggle.head(2).iterrows():
    tokens = ast.literal_eval(row['tokens'])
    labels = ast.literal_eval(row['labels'])
    pii_tokens = [(t, l) for t, l in zip(tokens, labels) if l != 'O']
    print(f'--- Kaggle record {i} ---')
    print(f'PII spans: {pii_tokens[:8]}\n')